In [ ]:
"""
DINOv2-Large + Attention U-Net Training Script
Attention Gates to Filter Irrelevant Features in Skip Connections

Key Feature: Attention gates help focus on relevant regions and reduce
background noise in skip connections.
"""
 
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from transformers import AutoModel
from PIL import Image
import numpy as np
import os
import csv
import time
from tqdm import tqdm
import matplotlib.pyplot as plt

plt.switch_backend('Agg')
torch.backends.cudnn.benchmark = True

# ============================================================================
# CONFIGURATION
# ============================================================================

CONFIG = {
    "image_size": (448, 448),
    "batch_size": 2,
    "accum_steps": 2,
    "lr": 8e-5,
    "weight_decay": 0.01,
    "epochs": 80,
    "num_classes": 10,
    "grad_clip": 1.0,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "train_path": "/kaggle/input/offroad-augmented-dataset/Offroad_Segmentation_Training_Augmented_Dataset/train",
    "val_path": "/kaggle/input/offroad-augmented-dataset/Offroad_Segmentation_Training_Augmented_Dataset/val",
    "output_dir": "/kaggle/working/dinov2_attention_unet_output",
    "resume_path": "/kaggle/input/last-wts-epoc-6-attn-unet-payload-offrd/last_checkpoint (1).pth"
}

# ============================================================================
# DATASET
# ============================================================================

class OffroadDataset(Dataset):
    def __init__(self, root_dir, transform=None, mask_transform=None):
        self.img_dir = os.path.join(root_dir, 'Color_Images')
        self.mask_dir = os.path.join(root_dir, 'Segmentation')
        self.files = sorted([f for f in os.listdir(self.img_dir) if f.endswith(('.png', '.jpg'))])
        self.transform = transform
        self.mask_transform = mask_transform
        self.value_map = {0:0, 100:1, 200:2, 300:3, 500:4, 550:5, 700:6, 800:7, 7100:8, 10000:9}
    
    def __len__(self):
        return len(self.files)
    
    def convert_mask(self, mask):
        mask_np = np.array(mask)
        new_mask = np.zeros_like(mask_np, dtype=np.uint8)
        for k, v in self.value_map.items():
            new_mask[mask_np == k] = v
        return Image.fromarray(new_mask)
    
    def __getitem__(self, idx):
        fname = self.files[idx]
        img = Image.open(os.path.join(self.img_dir, fname)).convert("RGB")
        mask = Image.open(os.path.join(self.mask_dir, fname))
        mask = self.convert_mask(mask)
        
        if self.transform:
            img = self.transform(img)
        if self.mask_transform:
            mask = self.mask_transform(mask) * 255
            
        return img, mask

# ============================================================================
# MODEL: Attention U-Net with DINOv2
# ============================================================================

class AttentionGate(nn.Module):
    """
    Attention Gate Module
    Filters out irrelevant features from skip connections
    
    Args:
        F_g: Channels from gating signal (decoder)
        F_l: Channels from encoder feature
        F_int: Intermediate channel dimension
    """
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, g, x):
        """
        Args:
            g: Gating signal from decoder (coarser scale)
            x: Encoder feature to be filtered (finer scale)
        """
        # Apply transformations
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        
        # Ensure same spatial size
        if g1.shape[2:] != x1.shape[2]:
            g1 = F.interpolate(g1, size=x1.shape[2:], mode='bilinear', align_corners=False)
        
        # Combine and generate attention coefficients
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        
        # Apply attention to encoder features
        return x * psi

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.conv(x)

class DinoAttentionUNet(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        
        # Frozen DINOv2-Large Encoder
        print("\n🔄 Downloading DINOv2-Large (2-3 mins on first run)...")
        import sys
        sys.stdout.flush()
        self.encoder = AutoModel.from_pretrained("facebook/dinov2-large")
        print("✅ DINOv2 loaded! Freezing encoder...")
        sys.stdout.flush()
        for param in self.encoder.parameters():
            param.requires_grad = False
        print("✅ Encoder frozen!\n")
        sys.stdout.flush()
        
        self.embed_dim = 1024
        
        # Decoder with Attention Gates
        self.bot = DoubleConv(self.embed_dim, 512)
        
        # Level 1
        self.up1 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.attn1 = AttentionGate(F_g=512, F_l=self.embed_dim, F_int=512)
        self.conv1 = DoubleConv(512 + self.embed_dim, 512)
        
        # Level 2
        self.up2 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.attn2 = AttentionGate(F_g=512, F_l=self.embed_dim, F_int=256)
        self.conv2 = DoubleConv(512 + self.embed_dim, 256)
        
        # Level 3
        self.up3 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.attn3 = AttentionGate(F_g=256, F_l=self.embed_dim, F_int=128)
        self.conv3 = DoubleConv(256 + self.embed_dim, 128)
        
        # Level 4
        self.up4 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.attn4 = AttentionGate(F_g=128, F_l=self.embed_dim, F_int=64)
        self.conv4 = DoubleConv(128 + self.embed_dim, 64)
        
        # Final upsampling and output
        self.final_up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.outc = nn.Conv2d(64, n_classes, 1)
    
    def forward(self, x):
        B, C, H, W = x.shape
        
        # Extract multi-scale features from DINOv2
        with torch.no_grad():
            outputs = self.encoder(x, output_hidden_states=True)
            hidden_states = outputs.hidden_states
            
            h_grid = H // 14
            w_grid = W // 14
            
            def reshape_feat(f):
                f = f[:, 1:, :]
                return f.permute(0, 2, 1).reshape(B, self.embed_dim, h_grid, w_grid)
            
            # Extract 4 encoder levels
            s1 = reshape_feat(hidden_states[7])   # Shallow
            s2 = reshape_feat(hidden_states[11])  # Mid-shallow
            s3 = reshape_feat(hidden_states[15])  # Mid-deep
            s4 = reshape_feat(hidden_states[23])  # Deep
        
        # Decoder with attention-gated skip connections
        x = self.bot(s4)
        
        # Level 1
        x = self.up1(x)
        # -----------------------------------------------------------------------
        s3 = F.interpolate(s3, size=x.shape[2:], mode='bilinear', align_corners=False) # <<<< ADD THIS
        # -----------------------------------------------------------------------
        s3_att = self.attn1(g=x, x=s3)
        x = torch.cat([x, s3_att], dim=1)
        x = self.conv1(x)
        
        # Level 2
        x = self.up2(x)
        # -----------------------------------------------------------------------
        s2 = F.interpolate(s2, size=x.shape[2:], mode='bilinear', align_corners=False) # <<<< ADD THIS
        # -----------------------------------------------------------------------
        s2_att = self.attn2(g=x, x=s2)
        x = torch.cat([x, s2_att], dim=1)
        x = self.conv2(x)
        
        # Level 3
        x = self.up3(x)
        # -----------------------------------------------------------------------
        s1 = F.interpolate(s1, size=x.shape[2:], mode='bilinear', align_corners=False) # <<<< ADD THIS
        # -----------------------------------------------------------------------
        s1_att = self.attn3(g=x, x=s1)
        x = torch.cat([x, s1_att], dim=1)
        x = self.conv3(x)
        
        # Level 4
        x = self.up4(x)
        s1_up = F.interpolate(s1, size=x.shape[2:], mode='bilinear', align_corners=False)
        s1_up_att = self.attn4(g=x, x=s1_up)
        x = torch.cat([x, s1_up_att], dim=1)
        x = self.conv4(x)
        
        # Final output
        x = self.final_up(x)
        logits = self.outc(x)
        
        if logits.shape[2:] != (H, W):
            logits = F.interpolate(logits, size=(H, W), mode='bilinear', align_corners=False)
        
        return logits

# ============================================================================
# METRICS
# ============================================================================

def compute_metrics(model, loader, device, num_classes):
    iou_scores, dice_scores, acc_scores = [], [], []
    model.eval()
    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            for imgs, labels in loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                labels = labels.squeeze(1).long()
                pred = torch.argmax(outputs, dim=1).view(-1)
                tgt = labels.view(-1)
                iou_cls, dice_cls = [], []
                for cls_id in range(num_classes):
                    p_inds, t_inds = pred == cls_id, tgt == cls_id
                    inter = (p_inds & t_inds).sum().float()
                    union = (p_inds | t_inds).sum().float()
                    if union > 0: 
                        iou_cls.append((inter/union).cpu().item())
                    dice_score = (2*inter + 1e-6)/(p_inds.sum() + t_inds.sum() + 1e-6)
                    dice_cls.append(dice_score.cpu().item())
                iou_scores.append(np.mean(iou_cls))
                dice_scores.append(np.mean(dice_cls))
                acc_scores.append((pred == tgt).float().mean().cpu().item())
    model.train()
    return np.mean(iou_scores), np.mean(dice_scores), np.mean(acc_scores)

def save_training_plots(hist, out_dir):
    plt.figure(figsize=(15, 5))
    plt.subplot(1, 3, 1)
    plt.plot(hist['train_loss'], label='Train')
    plt.plot(hist['val_loss'], label='Val')
    plt.title('Loss')
    plt.legend()
    plt.grid()
    
    plt.subplot(1, 3, 2)
    plt.plot(hist['val_iou'], label='Val IoU', color='orange')
    plt.title('IoU')
    plt.grid()
    
    plt.subplot(1, 3, 3)
    plt.plot(hist['lr'], label='Learning Rate', color='green')
    plt.title('LR Decay')
    plt.grid()
    
    plt.savefig(os.path.join(out_dir, 'training_curves.png'))
    plt.close()

# ============================================================================
# TRAINING LOOP
# ============================================================================

def train():
    os.makedirs(CONFIG['output_dir'], exist_ok=True)
    device = torch.device(CONFIG['device'])
    print(f"Using device: {device}")
    
    # Data
    h, w = CONFIG['image_size']
    transform = transforms.Compose([
        transforms.Resize((h, w)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    mask_transform = transforms.Compose([
        transforms.Resize((h, w), interpolation=transforms.InterpolationMode.NEAREST),
        transforms.ToTensor()
    ])
    
    train_ds = OffroadDataset(CONFIG['train_path'], transform, mask_transform)
    val_ds = OffroadDataset(CONFIG['val_path'], transform, mask_transform)
    train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
    
    # Model
    model = DinoAttentionUNet(CONFIG['num_classes']).to(device)
    
    # Optimizer & Scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
    total_steps = CONFIG['epochs'] * (len(train_loader) // CONFIG['accum_steps'])
    scheduler = torch.optim.lr_scheduler.PolynomialLR(optimizer, total_iters=total_steps, power=1.0)
    scaler = torch.amp.GradScaler('cuda')
    criterion = nn.CrossEntropyLoss(ignore_index=255)
    
    # Resume
    start_epoch = 0
    best_val_iou = 0.0
    history = {'train_loss': [], 'val_loss': [], 'val_iou': [], 'lr': []}
    
    ckpt_path = os.path.join(CONFIG['output_dir'], "last_checkpoint.pth")
    best_path = os.path.join(CONFIG['output_dir'], "best_model.pth")
    log_path = os.path.join(CONFIG['output_dir'], "training_log.csv")
    
    if CONFIG['resume_path'] and os.path.exists(CONFIG['resume_path']):
        print(f"Resuming from {CONFIG['resume_path']}...")
        checkpoint = torch.load(CONFIG['resume_path'], map_location=device, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        scaler.load_state_dict(checkpoint['scaler_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_val_iou = checkpoint.get('best_val_iou', 0.0)
        history = checkpoint.get('history', history)
        print(f"Resumed! Starting Epoch {start_epoch + 1}")
    
    if not os.path.exists(log_path):
        with open(log_path, 'w', newline='') as f:
            csv.writer(f).writerow(['Epoch', 'Train Loss', 'Val Loss', 'Val IoU', 'Val Dice', 'Val Acc', 'Time(s)', 'LR'])
    
    print(f"Training on {len(train_loader.dataset)} samples...")
    print(f"Config: BS={CONFIG['batch_size']}, Accum={CONFIG['accum_steps']} => Effective BS={CONFIG['batch_size'] * CONFIG['accum_steps']}")
    
    # Training Loop
    for epoch in range(start_epoch, CONFIG['epochs']):
        start_time = time.time()
        model.train()
        epoch_losses = []
        optimizer.zero_grad()
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['epochs']}", leave=False)
        for step, (imgs, masks) in enumerate(pbar):
            imgs, masks = imgs.to(device), masks.to(device)
            masks = masks.squeeze(1).long()
            
            with torch.amp.autocast('cuda'):
                outputs = model(imgs)
                loss = criterion(outputs, masks) / CONFIG['accum_steps']
            
            scaler.scale(loss).backward()
            
            if (step + 1) % CONFIG['accum_steps'] == 0:
                # Gradient clipping
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
                
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()
            
            epoch_losses.append(loss.item() * CONFIG['accum_steps'])
            current_lr = scheduler.get_last_lr()[0]
            pbar.set_postfix(loss=f"{loss.item() * CONFIG['accum_steps']:.4f}", lr=f"{current_lr:.6f}")
        
        # Validation
        val_losses = []
        model.eval()
        with torch.no_grad():
            with torch.amp.autocast('cuda'):
                for imgs, masks in val_loader:
                    imgs, masks = imgs.to(device), masks.to(device)
                    outputs = model(imgs)
                    loss = criterion(outputs, masks.squeeze(1).long())
                    val_losses.append(loss.item())
        
        mean_train_loss = np.mean(epoch_losses)
        mean_val_loss = np.mean(val_losses)
        val_iou, val_dice, val_acc = compute_metrics(model, val_loader, device, CONFIG['num_classes'])
        elapsed = time.time() - start_time
        final_lr = scheduler.get_last_lr()[0]
        
        history['train_loss'].append(mean_train_loss)
        history['val_loss'].append(mean_val_loss)
        history['val_iou'].append(val_iou)
        history['lr'].append(final_lr)
        
        print(f"\nEpoch {epoch+1} Summary:")
        print(f"  Train Loss: {mean_train_loss:.4f} | Val Loss: {mean_val_loss:.4f}")
        print(f"  Val IoU:    {val_iou:.4f}     | Val Dice: {val_dice:.4f} | LR: {final_lr:.6f}")
        
        with open(log_path, 'a', newline='') as f:
            csv.writer(f).writerow([epoch+1, mean_train_loss, mean_val_loss, val_iou, val_dice, val_acc, elapsed, final_lr])
        
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'best_val_iou': best_val_iou,
            'history': history
        }, ckpt_path)
        
        if val_iou > best_val_iou:
            print(f"  [!] New Best IoU ({best_val_iou:.4f} -> {val_iou:.4f}). Saving...")
            best_val_iou = val_iou
            torch.save(model.state_dict(), best_path)
        
        save_training_plots(history, CONFIG['output_dir'])
    
    print("\nTraining Finished!")
    print(f"Best Val IoU: {best_val_iou:.4f}")

if __name__ == "__main__":
    train()